<a href="https://colab.research.google.com/github/reshmiee/codemixing_llmrouter/blob/main/notebooks/Person02c_ScoreThirdSet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q routellm pandas openai openpyxl

In [4]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, time, hashlib
import numpy as np
import pandas as pd
os.chdir('/content/drive/MyDrive/ML_Research')

FOUNDRY_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = "placeholder"   # only so RouteLLM imports cleanly

from openai import OpenAI
import routellm.routers.matrix_factorization.model as mfmod
from routellm.controller import Controller

ENDPOINT = ""
mfmod.OPENAI_CLIENT = OpenAI(api_key=FOUNDRY_KEY, base_url=ENDPOINT)

client = Controller(
    routers=["bert", "mf"],
    strong_model="gpt-4-1106-preview",
    weak_model="mixtral-8x7b-instruct-v0.1",
)
bert, mf = client.routers["bert"], client.routers["mf"]
print("embedding model:", mf.model.embedding_model)

Mounted at /content/drive


config.json:   0%|          | 0.00/924 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  820kB            

model.safetensors: downloading bytes:           |  0.00B            

embedding model: text-embedding-3-small


In [5]:
DRY_RUN = False         # True = test on pilot, writes to _dryrun. False = real run.

# Real run: the MERGED third-set file (all 200 pairs), NOT one person's part.
REAL_IN_PATH  = "data/thirdset_v2.csv"             # the final merged 200-pair file (do not open or re-save it)
EXPECTED_SHA256 = "91d82d48a161527174fa82b47a9e041f8ad379ebbf7dc071147f05540b13a826"   # hash locked in the plan
EXPECTED_ROWS = 200                                 # real run refuses any other row count
REAL_OUT_PATH = "scores/scores_thirdset_v1.csv"
REAL_CFG_PATH = "results/config_thirdset_v1.json"

# Dry run: pilot file, outputs kept apart so they are never mistaken for results
DRY_IN_PATH  = "data/benchmark_v1.xlsx"
DRY_OUT_PATH = "_dryrun/scores_thirdset_dryrun.csv"
DRY_CFG_PATH = "_dryrun/config_thirdset_dryrun.json"

if DRY_RUN:
    IN_PATH, OUT_PATH, CFG_PATH = DRY_IN_PATH, DRY_OUT_PATH, DRY_CFG_PATH
else:
    IN_PATH, OUT_PATH, CFG_PATH = REAL_IN_PATH, REAL_OUT_PATH, REAL_CFG_PATH
for p in (OUT_PATH, CFG_PATH):
    os.makedirs(os.path.dirname(p), exist_ok=True)
print("DRY_RUN =", DRY_RUN, "| input:", IN_PATH, "| output:", OUT_PATH)

DRY_RUN = False | input: data/thirdset_v2.csv | output: scores/scores_thirdset_v1.csv


In [7]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def load_pairs(path):
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path, sheet_name="Benchmark" if "benchmark" in path else 0)
    return pd.read_csv(path)

def check_input(df, dry_run):
    assert {"pair_id", "english", "hinglish"} <= set(df.columns), df.columns.tolist()
    assert df["pair_id"].is_unique, "Duplicate pair_id"
    for c in ("english", "hinglish"):
        assert df[c].notna().all() and (df[c].astype(str).str.strip() != "").all(), f"Empty text in {c}"
    if not dry_run:
        assert "check_status" in df.columns, "No check_status column"
        bad = df[df["check_status"] != "OK"]
        assert bad.empty, f"{len(bad)} rows are not OK: {bad['pair_id'].tolist()[:10]}"
        assert len(df) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS} rows, got {len(df)}"

def score_both(df, cols=("english", "hinglish")):
    out = df.copy()
    for col in cols:
        out[f"{col}_bert"] = [bert.calculate_strong_win_rate(str(t)) for t in df[col]]
        out[f"{col}_mf"]   = [mf.calculate_strong_win_rate(str(t)) for t in df[col]]
    return out

In [8]:
df_test = load_pairs(DRY_IN_PATH).head(5)
print(score_both(df_test)[["pair_id","english_bert","hinglish_bert","english_mf","hinglish_mf"]])

   pair_id  english_bert  hinglish_bert  english_mf  hinglish_mf
0        1      0.249989       0.280707    0.139655     0.114239
1        2      0.205350       0.336730    0.090355     0.107800
2        3      0.467379       0.443534    0.282806     0.367780
3        4      0.282970       0.412477    0.083029     0.116454
4        5      0.234917       0.288684    0.176932     0.122855


In [12]:
if not DRY_RUN:
    assert not os.path.exists(OUT_PATH), f"{OUT_PATH} already exists. Score once only. Use a new version name and say so openly."

hash_before = sha256(IN_PATH)
print("sha256 before:", hash_before)
if not DRY_RUN:
    assert hash_before == EXPECTED_SHA256, f"Input hash does not match the locked plan: {hash_before}"
df = load_pairs(IN_PATH)
check_input(df, DRY_RUN)
print("rows:", len(df))

scored = score_both(df)
scored.to_csv(OUT_PATH, index=False)

hash_after = sha256(IN_PATH)
assert hash_before == hash_after, "Input file changed during scoring!"
print("saved", OUT_PATH, scored.shape)
print("sha256 after: ", hash_after)

sha256 before: 91d82d48a161527174fa82b47a9e041f8ad379ebbf7dc071147f05540b13a826
rows: 200
saved scores/scores_thirdset_v1.csv (200, 15)
sha256 after:  91d82d48a161527174fa82b47a9e041f8ad379ebbf7dc071147f05540b13a826


In [13]:
cfg = {
    "dry_run": DRY_RUN,
    "routers": ["bert", "mf"],
    "bert_checkpoint": "routellm/bert_gpt4_augmented (RouteLLM default; match config_conf_v1.json)",
    "strong_model": "gpt-4-1106-preview",
    "weak_model": "mixtral-8x7b-instruct-v0.1",
    "mf_embedding_model": "text-embedding-3-small (Azure-hosted deployment)",
    "azure_endpoint_host": "reshmiai901.openai.azure.com",
    "input_file": IN_PATH,
    "input_sha256": hash_before,
    "n_rows": int(len(df)),
    "output_file": OUT_PATH,
    "expected_sha256": EXPECTED_SHA256 if not DRY_RUN else None,
    "run_date": time.strftime("%Y-%m-%d %H:%M"),
    "routellm_version": __import__("importlib.metadata").metadata.version("routellm"),
}
json.dump(cfg, open(CFG_PATH, "w"), indent=2)
print(json.dumps(cfg, indent=2))

{
  "dry_run": false,
  "routers": [
    "bert",
    "mf"
  ],
  "bert_checkpoint": "routellm/bert_gpt4_augmented (RouteLLM default; match config_conf_v1.json)",
  "strong_model": "gpt-4-1106-preview",
  "weak_model": "mixtral-8x7b-instruct-v0.1",
  "mf_embedding_model": "text-embedding-3-small (Azure-hosted deployment)",
  "azure_endpoint_host": "reshmiai901.openai.azure.com",
  "input_file": "data/thirdset_v2.csv",
  "input_sha256": "91d82d48a161527174fa82b47a9e041f8ad379ebbf7dc071147f05540b13a826",
  "n_rows": 200,
  "output_file": "scores/scores_thirdset_v1.csv",
  "expected_sha256": "91d82d48a161527174fa82b47a9e041f8ad379ebbf7dc071147f05540b13a826",
  "run_date": "2026-11-15 08:11",
  "routellm_version": "0.2.0"
}


In [14]:
s = pd.read_csv(OUT_PATH)
cols = ["english_bert","hinglish_bert","english_mf","hinglish_mf"]
print(s[cols].describe().round(3))
print("missing:", int(s[cols].isna().sum().sum()), "| scores outside 0-1:", int(((s[cols] < 0) | (s[cols] > 1)).sum().sum()))
for r in ("bert", "mf"):
    print(f"{r.upper()} SD ratio (Eng/Hing): {s[f'english_{r}'].std() / s[f'hinglish_{r}'].std():.2f}")
if not DRY_RUN:
    assert len(s) == EXPECTED_ROWS and s[cols].notna().all().all() and ((s[cols] >= 0) & (s[cols] <= 1)).all().all(), "Sanity check failed"
    scores_sha256 = sha256(OUT_PATH)
    cfg["output_sha256"] = scores_sha256
    json.dump(cfg, open(CFG_PATH, "w"), indent=2)
    print("scores sha256:", scores_sha256)

       english_bert  hinglish_bert  english_mf  hinglish_mf
count       200.000        200.000     200.000      200.000
mean          0.499          0.480       0.199        0.217
std           0.167          0.084       0.079        0.079
min           0.103          0.266       0.064        0.073
25%           0.390          0.421       0.137        0.150
50%           0.499          0.482       0.199        0.215
75%           0.599          0.529       0.249        0.282
max           0.910          0.799       0.484        0.441
missing: 0 | scores outside 0-1: 0
BERT SD ratio (Eng/Hing): 2.00
MF SD ratio (Eng/Hing): 1.00
scores sha256: dd0bff026a0c87a26cd0c21b57bbcd92a07d5a6f4fc116719e51a0a14f61c6c4
